# Departamentos: selección de variables y PCA

Extracto de las actividades 2 y 3 del capstone no supervisado, asignadas a KFB. La actividad 1 y los datos preparados por el grupo no se incluyen. Las celdas dependen de variables definidas antes en el notebook completo. No se han vuelto a ejecutar ni se copian salidas.


## Actividad 2: categorías de precio y selección de variables


In [ ]:
# Tu código aquí ...
# Creamos una copia segura de features
features_categorized = features.copy()

# Generamos las categorías y los bins
categorias, bins = pd.qcut(features_categorized['price'], q=5, labels=['muy_bajo', 'bajo', 'mediano', 'alto','muy_alto'], retbins=True)
# Agregar la nueva columna al DataFrame
features_categorized['price_category'] = categorias
print(features_categorized[['price', 'price_category']].head())

# Mostrar la distribución de las categorías
print(features_categorized['price_category'].value_counts())


In [ ]:
# Preparando los datos
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Definir características (X) y variable objetivo (y)
X = features_categorized[['square_feet', 'bedrooms', 'Cluster']].dropna()
y = features_categorized['price_category']

# Convertir y a valores numéricos
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Escalar X (para métodos que lo requieran)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

# Aplicar Chi-cuadrado (solo en valores no negativos, por eso no escalamos)
chi2_selector = SelectKBest(chi2, k=2)
X_chi2 = chi2_selector.fit_transform(X, y)

# Características seleccionadas
selected_features_chi2 = X.columns[chi2_selector.get_support()]
print("Características seleccionadas por Chi-cuadrado:", selected_features_chi2.tolist())

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Definir el modelo base (Regresión Logística)
log_reg = LogisticRegression(max_iter=1000)

# Aplicar RFE para seleccionar 2 características
rfe_selector = RFE(log_reg, n_features_to_select=2)
rfe_selector.fit(X_scaled, y_encoded)  # Aquí sí usamos los datos escalados

# Obtener las características seleccionadas
selected_features_rfe = X.columns[rfe_selector.support_]
print("Características seleccionadas por RFE:", selected_features_rfe.tolist())


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Entrenar un modelo de Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_scaled, y_encoded)

# Obtener importancia de características
feature_importances = rf_model.feature_importances_
selected_features_rf = X.columns[np.argsort(feature_importances)[-2:]]  # Seleccionamos las 2 más importantes

print("Características seleccionadas por Random Forest:", selected_features_rf.tolist())


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Definir el conjunto de características seleccionadas
X_selected = X[['square_feet', 'Cluster']]  # Usamos las seleccionadas por los tres métodos

# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_encoded, test_size=0.2, random_state=42)

# Entrenar modelo de Árbol de Decisión
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Hacer predicciones
y_pred = model.predict(X_test)

# Evaluar el modelo
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


In [ ]:
# Entrenar el modelo usando TODAS las características
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Entrenar Árbol de Decisión con todas las características
model_full = DecisionTreeClassifier(random_state=42)
model_full.fit(X_train_full, y_train_full)

# Hacer predicciones
y_pred_full = model_full.predict(X_test_full)

# Evaluar el modelo con todas las características
print("Reporte de Clasificación (Todas las características):")
print(classification_report(y_test_full, y_pred_full, target_names=label_encoder.classes_))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extraer datos con latitud, longitud y categoría de precio
geo_data = features_categorized[['latitude', 'longitude', 'price_category']].dropna()

# Definir una paleta de colores personalizada (azul → rojo)
palette = {
    'muy_bajo': 'blue',
    'bajo': 'cyan',
    'mediano': 'green',
    'alto': 'orange',
    'muy_alto': 'red'
}

# Crear scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=geo_data['longitude'],
    y=geo_data['latitude'],
    hue=geo_data['price_category'],  # Colorear por categoría de precio
    palette=palette,
    alpha=0.5,  # Transparencia para mejor visualización
    edgecolor=None
)

# Etiquetas y título
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("Distribución de Ubicaciones con Categoría de Precio")
plt.legend(title="Categoría de Precio", bbox_to_anchor=(1.05, 1), loc='upper left')  # Mover leyenda fuera del gráfico
plt.show()


In [ ]:
import plotly.express as px

# Crear gráfico 3D interactivo
fig = px.scatter_3d(
    features_categorized,
    x='longitude',
    y='latitude',
    z='square_feet',  # Usamos el tamaño del departamento como tercera dimensión
    color=features_categorized['price_category'],  # Color según la categoría de precio
    color_discrete_map={  # Definir colores personalizados para cada categoría
        'muy_bajo': 'blue',
        'bajo': 'cyan',
        'mediano': 'green',
        'alto': 'orange',
        'muy_alto': 'red'
    },
    title="Visualización 3D: Ubicación y Categoría de Precio"
)

# Mostrar el gráfico interactivo
fig.show()


## Actividad 3: PCA y clasificación


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Seleccionar las características numéricas originales antes de aplicar PCA
X_pca = features_categorized[['square_feet', 'Cluster']]

# Normalizar los datos
scaler = StandardScaler()
X_pca_scaled = scaler.fit_transform(X_pca)

# Verificar dimensiones antes de PCA
print("Dimensión de los datos antes de PCA:", X_pca_scaled.shape)


In [ ]:
# Aplicar PCA con 2 componentes principales
pca = PCA(n_components=2)
X_pca_reduced = pca.fit_transform(X_pca_scaled)

# Convertir a DataFrame
X_pca_df = pd.DataFrame(X_pca_reduced, columns=['PC1', 'PC2'])
X_pca_df['price_category'] = y_encoded[:X_pca_df.shape[0]]  # Agregar la variable objetivo

# Verificar la nueva estructura
print("Dimensión de los datos después de PCA:", X_pca_df.shape)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear scatter plot de PC1 vs PC2 con color según la categoría de precio
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=X_pca_df['PC1'],
    y=X_pca_df['PC2'],
    hue=X_pca_df['price_category'],
    palette='viridis',
    alpha=0.7
)

# Etiquetas y título
plt.xlabel("Principal Component 1 (PC1)")
plt.ylabel("Principal Component 2 (PC2)")
plt.title("Distribución de los Datos después de PCA")
plt.legend(title="Categoría de Precio", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


In [ ]:
# Dividir datos en entrenamiento y prueba
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca_df[['PC1', 'PC2']], y_encoded[:X_pca_df.shape[0]], test_size=0.2, random_state=42)

# Entrenar el modelo con las componentes principales generadas por PCA
model_pca = DecisionTreeClassifier(random_state=42)
model_pca.fit(X_train_pca, y_train_pca)

# Hacer predicciones
y_pred_pca = model_pca.predict(X_test_pca)

# Evaluar el modelo con PCA
print("Reporte de Clasificación (Con PCA):")
print(classification_report(y_test_pca, y_pred_pca, target_names=label_encoder.classes_))
